**MÓDULO 04 - 1.4_F - PROJETO - ENRIQUECENDO DADOS PARA CAMADA SILVER 1**

Para fazermos a tradução daqueles arquivos que estamos utilizando como csv, que representam o export de dados desse SAP, quando a gente encontra cenários como esse onde a coluna é representada por código, o nome da tabela também é representada por código. Todas essas características precisamos traduzir todas elas já na acada silver. 
Porem, precisamos de um de para, do código para a descrição o que isso significa. Para a gente conseguir realizar esse tipo de atividade, a gente precisa de um glossário. Esse glossário ele possui essa lógica, cada um tem um formato um padrão diferente mas o objetivo é o mesmo. 
Então aqui vamos identificar a tabela na sua origem o nome dela, o que ela significa e também a dercrição de cada uma das colunas, cada código de cada coluna tem de significado. 

→ Quando levamos isso para a camada silver eu não posso mais adotar o padrão da orgiem da origem, eu levo trago isso para um modelo padronizado para um ambiente analítico. Então eu não posso identificar um produto por um código de uma determinada aplicação. Imagina se eu utilizo 2 ou 3 aplicações ou tenho filiais diferentes e cada uma dessas filiais utilizam sistemas diferentes. Então aqui eu não posso ficar utilizando padrões diferentes.

→ Eu posso fazer a ingestão isso na camada bronze, posso seguir esse padrão mas quando eu trago para silver eu já disponibilizo esse dado para consultas analíticas. 
Então o ID de produto, nome de produto, descrição, categoria, todos esses itens e atributos, eu já preciso trazer normalizado e unificado na minha camada silver. 

→ Então aqui eu posso ter 2 ou mais aplicações e cada uma seguindo um padrão diferente, mas quando eu faço a ingestão para a camada silver, esses dados precisam conversar. Então aqui na camada silver a gente já aplica esses padrões.

•	Então vamos dar uma olhada como traduzimos aquela ingestão para a camada silver.


In [1]:
# Aqui o professor abriu um novo arquivo jupyter notebook e salvou na pasta scripts do prejeto no diretório do projeto GitHub.
# Vamos importat a bilbioteca duckdb para trabalhar com o banco de dados e o pandas para trabalhar com os dataframes.
import duckdb
import pandas as pd

In [2]:
# Vamos abrir a conexão com o banco de dados, utilizando o comando duckdb.connect() para criar a conexão com o banco de dados.
# Lembrando que estamos utilizando o banco de dados que criamos anteriormente, que é o dados_duckdb.db, e o parâmetro read_only=False para permitir que possamos escrever no banco de dados.
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [3]:
# Agora podemos recuperar os da tabela que criamos anteriormente, que é a tabela bronze_z0019, utilizando o comando con.execute() para fazer a consulta nessa tabela bronze_z0019 e depois utilizando o comando .fetchdf() para trazer o resultado dessa consulta como um dataframe.
# Coloquei o resultado do select dentro de uma variável chamada df, que é a convenção para nomear dataframes, e depois utilizei o comando .head(10) para mostrar as 10 primeiras linhas desse dataframe.
df = con.execute('SELECT * FROM bronze_z0019').fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-03-04 10:22:55.262189
1,10002,MARTELO,BT50,100,500,z0019_1.csv,2026-03-04 10:22:55.262189
2,10003,PREGO,BT10,100,50,z0019_1.csv,2026-03-04 10:22:55.262189
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-03-04 10:40:35.014454
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-03-04 10:40:35.014454
5,10003,PREGO,BT10,100,50,z0019_2.csv,2026-03-04 10:40:35.014454


In [4]:
# Repare que aqui eu tenho alguns registros com o ID identificador repetido.
# E esses códigos que estão nas colunas, já possuímos o glossário e temos o nome o significado de cada coluna.
# Então aqui também já podemos substituir esses códigos pelos seus significados.
# Para esse caso iremos utilizar o SQL para pegar o último registro por ID.
# Para fazer isso a gente vai ordernar os registros com BASE NA DATA DE INGESTÃO e aí a gente vai identificar qual é o último
# e aí sim fazemos essa alteração.
# Aqui eu tenho dois registros que terminam com 10003 e o nome dele da descrição é o PREGO e temos o preço de 50 em um e 60 no outro para o mesmo
# Então perceba que são de arquivos diferentes, mas o mesmo produto, e a data a gente consegue ordernar que o ultimo dado vai ser esse 2026-03-04 10:40:35.014454
# Então eu não preciso ter essa informação antiga do prego que está lá em cima

# Então aqui vamos ter que utilizar alguns recursos do SQL para fazer essa consulta.
# Vou efetuar a consulta filtrando a data de ingestão para pegar somente os registros que foram ingeridos a partir do dia 11 de janeiro de 2025, para pegar somente os registros mais recentes.
# Não teve nenhuma alteração na tabela, por que ps registros são todos de hoje.

df = con.execute('''
                SELECT * 
                FROM 
                bronze_z0019 
                where data_ingestao >='2025-01-11'
                ''').fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-03-04 10:22:55.262189
1,10002,MARTELO,BT50,100,500,z0019_1.csv,2026-03-04 10:22:55.262189
2,10003,PREGO,BT10,100,50,z0019_1.csv,2026-03-04 10:22:55.262189
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-03-04 10:40:35.014454
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-03-04 10:40:35.014454
5,10003,PREGO,BT10,100,50,z0019_2.csv,2026-03-04 10:40:35.014454


In [5]:
# Após a consulta anterior vamos adicionar um outro registro no comando de consulta
# Vamos adicionar o número da linha utilizando a função ROW_NUMBER() para numerar as linhas do resultado da consulta, 
# e vamos particionar essa numeração por ID, para que a numeração seja reiniciada para cada ID diferente, 
# e vamos ordernar essa numeração pela data de ingestão em ordem decrescente, para que o registro mais recente receba o número 1.   
# Aqui ele vai adicionar um número de forma incremental com base no ID e pela data de ingestão, 
# e aí a gente vai filtrar somente os registros que tem o número 1, ou seja, os registros mais recentes para cada ID.  
# Ele vai agrupar pelos IDs e vai ordernar pela data de ingestão, e aí a gente vai pegar somente o registro mais recente para cada ID. 
# Ele vai colocar um incremental com base na data de ingestão, e aí a gente vai pegar somente o registro mais recente para cada ID.

df = con.execute('''                
                SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row   
                FROM 
                bronze_z0019 
                where data_ingestao >='2025-01-11'
                ''').fetchdf()
df.head(10)

# veja que ambos ficaram com o row 1 e somente 1 regristro ficou com o row 2
# ele ficou assim por que ele vai agrupar por NATBR e aí ele vai ordernar pela data de ingestão,
# e aí ele vai colocar o número 1 para o registro mais recente, e o número 2 para o registro mais antigo, e assim por diante. 
# os que foram inseridos por último, recebeu o ID 1   
# o que foi inserido anteriormente a ele recebeu o ID 2
# Logo podemos concluir que, os registros mais atualizados para cada ID são os que tem o row 1, e os registros mais antigos para cada ID são os que tem o row 2.

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10004,SERRA,BT50,100,200,z0019_2.csv,2026-03-04 10:40:35.014454,1
1,10002,MARTELO,BT50,100,500,z0019_1.csv,2026-03-04 10:22:55.262189,1
2,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-03-04 10:40:35.014454,1
3,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-03-04 10:22:55.262189,1
4,10003,PREGO,BT10,100,50,z0019_2.csv,2026-03-04 10:40:35.014454,1
5,10003,PREGO,BT10,100,50,z0019_1.csv,2026-03-04 10:22:55.262189,2


In [6]:
# Então aqui a gente pode filtrar somente os registros que tem o row 1, para pegar somente os registros mais recentes para cada ID.
# Aqui estamos utilizando um outro conceito que é o INNER SELECT ou o select aninhado, onde a gente faz um select dentro de outro select, 
# para poder filtrar os resultados do select interno.
# no select interno a gente faz a consulta para pegar o número da linha para cada registro, 
# e aí no select externo a gente filtra somente os registros que tem o número 1, ou seja, os registros mais recentes para cada ID.

df = con.execute('''        
                SELECT * FROM (
                    SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row   
                    FROM 
                    bronze_z0019 
                    where data_ingestao >='2025-01-11'
                ) WHERE row = 1
                ''').fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10003,PREGO,BT10,100,50,z0019_2.csv,2026-03-04 10:40:35.014454,1
1,10004,SERRA,BT50,100,200,z0019_2.csv,2026-03-04 10:40:35.014454,1
2,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-03-04 10:40:35.014454,1
3,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-03-04 10:22:55.262189,1
4,10002,MARTELO,BT50,100,500,z0019_1.csv,2026-03-04 10:22:55.262189,1


In [7]:
# Agora podemos realizaar outras alterações nesse dataframe, como por exemplo,
# substituir os códigos pelos seus significados utilizando o comando CASE WHEN do SQL, para substituir os códigos pelos seus significados.
# Que alterações são essas que podemos fazer? A coluna nome_arquivo, data_ingestao, row esses três eu não preciso mais.
# As outras colunas que estão com nome de código, vimos no glossário o significado de cada uma delas.

# Primeiro eu vou deletar as colunas que eu não preciso mais, utilizando o comando DROP COLUMN do SQL, para deletar as colunas nome_arquivo, data_ingestao e row.
# Depois eu vou substituir os códigos pelos seus significados utilizando o comando CASE WHEN do SQL, para substituir os códigos pelos seus significados.
# Aqui não estou alerando o dataframe df, estou deletand as colunas e estou criando um novo dataframe com as alterações que eu quero
# Estou criando um outro dataframe para não perder o dataframe original, caso eu queira voltar para ele depois.

df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row'])
df_final.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST
0,10003,PREGO,BT10,100,50
1,10004,SERRA,BT50,100,200
2,10005,MACHADO,BT50,100,100
3,10001,PARAFUSO,BT10,100,100
4,10002,MARTELO,BT50,100,500


In [8]:
# Agora com base no Glossário
# Vamos renomear os nomes das colunas utilizando o comando RENAME COLUMN do SQL, para renomear as colunas com os seus significados.
# Aqui alterei o nome das colunas para ficar mais fácil de entender o que cada coluna representa, e para ficar mais legível o dataframe. 
# Aqui podemos seguir esse padrão do nome das colunas com base no glossário, ou em alguns outros contextos podemos seguir 
# alguns dos outros padrões com base em prefixos EX: no_produtos isso seria o nome da coluna nome do produto, ou no_fornecedor, isso seria o nome da coluna nome do fornecedor, e assim por diante.
# Aqui a gente tem um código então seria o ID da categoria
# Fornecedor também é um código então é um id_fornecedor
# E o preço seria um valor então posso colocar vl_preco
df_final = df_final.rename(columns={'NATBR': 'id'})   
df_final = df_final.rename(columns={'MAKTX': 'nm_produto'})
df_final = df_final.rename(columns={'WERKS': 'id_categoria'})
df_final = df_final.rename(columns={'MAINS': 'id_fornecedor'})
df_final = df_final.rename(columns={'LABST': 'vl_preco'})
df_final.head(10)
# Pronto! DataFrame estruturado!


,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10003,PREGO,BT10,100,50
1,10004,SERRA,BT50,100,200
2,10005,MACHADO,BT50,100,100
3,10001,PARAFUSO,BT10,100,100
4,10002,MARTELO,BT50,100,500


Agora precisamos fazer a ingestão desses dados, mas antes precisamos enriquecer um pouquinho mais esse dataframe.


**MÓDULO 04 - 1.4_G - PROJETO - ENRIQUECENDO DADOS PARA CAMADA SILVER 2**

No último vídeo a gente organizou o nosso dataframe já com o nome das colunas que queremos e com os nomes que queremos. 

→ Vamos continuar trabalhar com o df_final

In [9]:
# Vamos trocar agora o df_final para o df2, então o df2 vai ser uma cópia do df_final
# E aqui vamos conseguir fazer mais algumas alterações.
df2 = df_final.copy()
df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10003,PREGO,BT10,100,50
1,10004,SERRA,BT50,100,200
2,10005,MACHADO,BT50,100,100
3,10001,PARAFUSO,BT10,100,100
4,10002,MARTELO,BT50,100,500


In [11]:
# Depois de copiar, agora df2 ai ser igual a df2.e aqui iremos definir os tipos de cada uma das colunas
# Mas para isso vamos dar uma olhada antes como está tipada cada uma dessas colunas, para isso vamos utilizar o comando .dtypes para verificar
# os tipos de cada uma das colunas do dataframe df_final.
# Aqui mostra que está tudo como 'objeto', ou seja, tudo como string, tudo aqui é texto.
# Se precisarmos fazer algum cálculo com o preço, alguma ordenação por preço ou por valor,
# é interessante que a coluna de preço esteja com o tipo numérico, 
# para isso vamos alterar o tipo dessa coluna para numérico utilizando o comando pd.to_numeric() do pandas, para converter a coluna vl_preco para numérico.
df_final.dtypes

id               str
nm_produto       str
id_categoria     str
id_fornecedor    str
vl_preco         str
dtype: object

In [12]:
# Fazendo a conversão da coluna vl_preco para numérico utilizando o comando pd.to_numeric() do pandas, para converter a coluna vl_preco para numérico.
df2 = df_final.copy()
df2['vl_preco'] = pd.to_numeric(df2['vl_preco'], errors='coerce')
df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10003,PREGO,BT10,100,50
1,10004,SERRA,BT50,100,200
2,10005,MACHADO,BT50,100,100
3,10001,PARAFUSO,BT10,100,100
4,10002,MARTELO,BT50,100,500


In [17]:
# Posso efetuar a alteração da forma que o professor fez.
# Após inserir a função 'astype' para alteração de tipos das colunas, aqui passeo um objeto com as chaves DICIONÁRIOS são OBJETOS aqui em Python
# Passei o nome de cada coluna e o valor com o tipo
# No id vai ter um inteiro32, nm_produto e id_categoria vai ser do tipo string, id_fornecedor vai ser inteiro32 e vl_preco vai ser float32

df2 = df_final.copy()
df2 = df2.astype(
            {
                'id': 'int32',
                'nm_produto': 'string',
                'id_categoria': 'string',
                'id_fornecedor': 'int32',
                'vl_preco': 'float32'  
            }
)


In [13]:
# Só para explicar temos outras formas de fazer a alteração do tipo das colunas,
# como por exemplo utilizando o comando astype() do pandas, utilizando a tipagem do python, ou seja, utilizando o tipo int, str, 
# float e assim por diante.

df2 = df_final.copy()
df2 = df2.astype(
            {
                'id': int,
                'nm_produto': str,
                'id_categoria': str,
                'id_fornecedor': int,
                'vl_preco': float  
            }
)
df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10003,PREGO,BT10,100,50.0
1,10004,SERRA,BT50,100,200.0
2,10005,MACHADO,BT50,100,100.0
3,10001,PARAFUSO,BT10,100,100.0
4,10002,MARTELO,BT50,100,500.0


In [14]:
# Mas para a gente armazenar esses dados, precisamos criar a tabela.
# Para isso vamos utilizar a conexão e a execução de comandos SQL para criar a tabela e inserir os dados do dataframe df2 nessa tabela.
# Agora já posso trazer o nome da tabela, o que ela significa, no caso 'produtos'
con.execute('''
    CREATE TABLE IF NOT EXISTS produtos (
        id BIGINT,
        nm_produto TEXT,
        id_categoria TEXT,
        id_fornecedor BIGINT,
        vl_preco FLOAT
    )
''')
# Agora sim eu tenho onde inserir esses registros aqui.


# Vamos inserir o df2 que é o dataframe que a gente acabou de criar, com as alterações que a gente fez, para dentro da tabela produtos utilizando o comando to_sql() do pandas, para inserir os dados do dataframe df2 na tabela produtos do banco de dados.
# Vamos dar uma olhada no df2
df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10003,PREGO,BT10,100,50.0
1,10004,SERRA,BT50,100,200.0
2,10005,MACHADO,BT50,100,100.0
3,10001,PARAFUSO,BT10,100,100.0
4,10002,MARTELO,BT50,100,500.0


In [18]:
# Vamos consultar o conteúdo da tabela produtos para verificar se os dados foram inseridos corretamente, utilizando
# o comando con.execute() para fazer a consulta na tabela produtos e depois utilizando o comando .fetchdf()
# para trazer o resultado dessa consulta como um dataframe.
# Certamente está vazia, poi somente criei a tabela, mas não inseri os dados ainda.

df_resultado = con.execute('SELECT * FROM produtos').fetchdf()
df_resultado.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10002,MARTELO,BT50,100,500.0
1,10001,PARAFUSO,BT10,100,100.0
2,10003,PREGO,BT10,100,50.0
3,10005,MACHADO,BT50,100,100.0
4,10004,SERRA,BT50,100,200.0
5,10001,PARAFUSO,BT10,100,100.0
6,10004,SERRA,BT50,100,200.0
7,10003,PREGO,BT10,100,50.0
8,10002,MARTELO,BT50,100,500.0
9,10005,MACHADO,BT50,100,100.0


In [21]:
# Aqui temos as tabelas vazias, pois somente criamos a tabela, mas não inserimos os dados ainda.
# Agora vamos inserir os dados do dataframe df2 na tabela produtos utilizando o comando to_sql()
con.execute('''insert into produtos SELECT * FROM df2''')

In [19]:
# Agora eu posso fechar a conexão com o banco de dados utilizando o comando con.close() para fechar a conexão com o banco de dados.
con.close()

1. Então aqui já trouxe os dados que eu recebi na landing para a Bronze.
2. Enriqueci eles na Silver.
3. E agora vamos levar esse exemplo simples para a camada Gold.
4. Vamos fazer isso separadamente em um outro Notebook pois o mesmo é esécífico da camada Gold.